# 📊 Müşteri Kaybı (Churn) Tahmin Projesi
---

## 🎯 1. Proje Amacı ve İçeriği
Bu çalışma, bir telekomünikasyon şirketinin müşteri verilerini analiz ederek, hangi müşterilerin hizmet almayı bırakacağını (**Churn**) önceden tahmin etmek amacıyla geliştirilmiştir.

**Veri Kaynağı:** Bu projede kullanılan veri seti [Kaggle - Telco Customer Churn (IBM)](https://www.kaggle.com/datasets/yeanzc/telco-customer-churn-ibm-dataset) üzerinden temin edilmiştir.

### **Proje Kapsamı:**
* **Veri Analizi:** Müşteri demografisi ve kullanım alışkanlıklarının incelenmesi.
* **Veri Ön İşleme:** Eksik verilerin temizlenmesi ve kategorik verilerin sayısallaştırılması.
* **Model Eğitimi:** **TensorFlow/Keras** ile derin öğrenme tabanlı bir Yapay Sinir Ağları (ANN) kurulması.
* **Optimizasyon:** Yapılan analizler sonucunda modelin ezberlemesini (overfitting) önlemek adına eğitimin **25 epoch** ile sınırlandırılması.
* **Tahminleme:** Mevcut verilerle gelecekteki müşteri davranışlarının öngörülmesi.

---

## 🛠 2. Kullanılan Teknolojiler
* **Veri İşleme:** `Pandas`, `Numpy`
* **Makine Öğrenmesi:** `Scikit-Learn` (Scaling & Splitting)
* **Derin Öğrenme:** `TensorFlow & Keras` (Yapay Sinir Ağları)
* **Görselleştirme:** `Matplotlib`, `Seaborn`

---

## ⚠️ 3. Sistem ve Kurulum Gereksinimleri
Projenin sorunsuz çalışması için **Python 3.13** versiyonu önerilmektedir. Lütfen aşağıdaki adımları takip ederek ortamınızı hazırlayın:

### **A. Python Versiyon Kontrolü ve Kurulumu**
Terminal veya CMD (Komut İstemi) üzerinden sisteminizdeki Python versiyonlarını kontrol etmek için:
`py -0`

* **Eğer 3.13 kurulu değilse:** [Buraya tıklayarak Python 3.13.7 (64-bit) sürümünü indirebilirsiniz.](https://www.python.org/ftp/python/3.13.7/python-3.13.7-amd64.exe)

### **B. Gerekli Paketlerin Kurulması**
Python 3.13 kurulduktan sonra, tüm kütüphaneleri ve Jupyter Notebook'u bu versiyona yüklemek için CMD'ye şu komutu yapıştırın:
```bash
py -3.13 -m pip install notebook tensorflow pandas numpy scikit-learn matplotlib seaborn joblib

## 🔍 1. Veri Seti Hazırlığı ve Temizliği
Bu aşamada verideki eksik değerleri kontrol ediyor ve model için anlam taşımayan **ID** gibi sütunları temizliyoruz. 
> **Not:** Sayısal olması gereken `TotalCharges` sütunundaki hatalı formatlar bu bölümde düzeltilmiştir.

In [1]:
import pandas as pd
import numpy as np

# Veri setini okuyoruz
# '../data/' ifadesi: bir üst klasöre çık ve data klasörüne gir demektir
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# İlk 5 satırı kontrol edelim
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
# Veri tiplerini ve eksik değerleri kontrol et
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [3]:
# customerID sütununu siliyoruz
df.drop('customerID', axis=1, inplace=True)
print("Sütun silindi. Yeni sütun sayısı:", len(df.columns))

Sütun silindi. Yeni sütun sayısı: 20


In [4]:
# Boşluk içeren satırları bulup onları NaN (boş) yapıyoruz, sonra sütunu sayıya çeviriyoruz
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Boş kalan satırları (NaN) temizliyoruz
df.dropna(inplace=True)

print("Veri sayıya çevrildi ve boş satırlar temizlendi.")

Veri sayıya çevrildi ve boş satırlar temizlendi.


## 🔢 2. Verilerin Sayısallaştırılması
Bilgisayarın metinleri işleyebilmesi için **One-Hot Encoding** yöntemi kullanılarak kategorik değişkenler sayısal verilere dönüştürülmüştür.

In [5]:
# 'Churn' (ayrılma durumu) sütununu 1 ve 0 yapalım
df['Churn'] = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

# Diğer tüm metin sütunlarını otomatik olarak sayısal verilere (kukla değişkenlere) çevirelim
df_final = pd.get_dummies(df)

# Son halini kontrol et
df_final.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,...,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,True,False,False,True,True,...,False,True,False,False,False,True,False,False,True,False
1,0,34,56.95,1889.50,0,False,True,True,False,True,...,False,False,True,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,False,True,True,False,True,...,False,True,False,False,False,True,False,False,False,True
3,0,45,42.30,1840.75,0,False,True,True,False,True,...,False,False,True,False,True,False,True,False,False,False
4,0,2,70.70,151.65,1,True,False,True,False,True,...,False,True,False,False,False,True,False,False,True,False


## 📉 3. Veriyi Bölme ve Ölçeklendirme (Min-Max Scaling)

Bu aşama, modelin kararlı çalışması için iki kritik işlem içerir:

* **Eğitim ve Test Ayırımı:** Veri setini %80 oranında eğitim, %20 oranında test olacak şekilde bölüyoruz.
* **Veri Ölçeklendirme (Scaling):** Verilerimizdeki farklı sayısal aralıkları (örneğin; aylık ücretler ile kullanım süresi) `0` ile `1` arasına sıkıştırıyoruz.
    * **Önemli:** `fit_transform` fonksiyonu ile eğitim setindeki kuralı öğreniyor, `transform` ile bu kuralı test setine uyguluyoruz.

> **Not:** Bu işlem yapılmazsa, büyük sayısal değerler model üzerinde haksız bir ağırlık oluşturarak tahminleri bozabilir.

In [6]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# 1. Veriyi Giriş (X) ve Hedef (y) olarak ayıralım
X = df_final.drop('Churn', axis=1)
y = df_final['Churn']

# 2. Veriyi Eğitim ve Test olarak ayıralım (%80 - %20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Ölçeklendirme
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Veri %80 Eğitim ve %20 Test olarak ayarlandı!")

Veri %80 Eğitim ve %20 Test olarak ayarlandı!


## 🧠 4. Yapay Sinir Ağı (ANN) Mimarisi
Modelimiz, verideki gizli desenleri öğrenmek için ardışık katmanlardan oluşur:
1.  **Giriş Katmanı:** 45 parametre girişi.
2.  **Gizli Katmanlar:** Karmaşıklığı çözmek için 20 ve 15 nöronlu `ReLU` katmanları.
3.  **Çıkış Katmanı:** `Sigmoid` aktivasyonu ile 0-1 arası olasılık tahmini.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# 1. Modelin Mimarisini Kuruyoruz
from tensorflow.keras.layers import Input # Input katmanını içeri aktar

model = Sequential([
    Input(shape=(45,)),              # Önce giriş boyutunu ayrı bir katman olarak tanımla
    Dense(15, activation='relu'),    # Sonra nöronları ekle
    Dense(5, activation='relu'),
    Dense(1, activation='sigmoid')
])

# 2. Modeli Derliyoruz
model.compile(optimizer='adam', 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

# 3. Eğitimi Başlatıyoruz (50 Epoch)
history = model.fit(X_train, y_train, epochs=50, validation_split=0.2)

Epoch 1/50


In [ ]:
# 1. Test verileriyle modeli değerlendirelim
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Doğruluğu: %{test_acc*100:.2f}")

# 2. İlk 5 test örneği için tahmin yapalım
predictions = model.predict(X_test)

# Tahminleri (olasılıkları) 1 ve 0'a çevirelim
y_pred = [1 if p > 0.5 else 0 for p in predictions]

print("İlk 5 Tahmin:", y_pred[:5])
print("Gerçek Değerler:", y_test[:5].values)

## 📈 5. Model Eğitim Performansı
Aşağıdaki grafikler, modelin her adımda (**Epoch**) ne kadar geliştiğini, hata oranının nasıl düştüğünü göstermektedir.

In [ ]:
import matplotlib.pyplot as plt

# Doğruluk (Accuracy) grafiği
plt.plot(history.history['accuracy'], label='Eğitim Başarısı')
plt.plot(history.history['val_accuracy'], label='Doğrulama Başarısı')
plt.title('Model Başarısı')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## 🕵️ Model Değerlendirmesi: En Optimal Nokta Analizi

Yukarıdaki başarı ve hata (Accuracy & Loss) grafiklerini incelediğimizde, modelin performansı için **25. epoch** civarının en optimal nokta (Sweet Spot) olduğu gözlemlenmiştir.

**Neden 25. Epoch?**
* **Denge:** Bu noktada Eğitim (Train) ve Doğrulama (Validation) başarıları birbirine en yakın seviyededir.
* **Genelleme Yeteneği:** 25. epoch'tan sonra mavi çizgi yükselmeye devam etse de turuncu çizginin sabitlendiği görülmektedir. Bu, modelin bu noktadan sonra veriyi öğrenmek yerine yavaş yavaş "ezberlemeye" (**Overfitting**) başladığının bir işaretidir.
* **Karar:** Modelin gerçek dünya verilerinde (hiç görmediği müşterilerde) en yüksek isabetle çalışması için 25-30 epoch arası bir eğitimin bu veri seti için en sağlıklı sonuç olduğu sonucuna varılmıştır.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek Değer')
plt.title('Hata Matrisi')
plt.show()

## 💾 6. Modelin Dışa Aktarılması
Eğitilen model, ileride gerçek zamanlı tahminler yapmak üzere `churn_modeli.keras` adıyla kaydedilmiştir.

In [ ]:
model.save('../models/churn_modeli.keras')
print("Model en güncel formatta kaydedildi!")

## 💾 7. Model ve Ölçeklendirme Anahtarının Kaydedilmesi

Projenin en önemli adımı burasıdır. Sadece modeli değil, veriyi modele uygun hale getiren **Scaler** nesnesini de kaydediyoruz:

* **Model Kaydı:** Eğitilen yapay zeka beynini `.keras` formatında saklıyoruz.
* **Scaler Kaydı:** Yeni gelecek verileri, modelin anladığı 0-1 arasına dönüştürmek için kullandığımız "ölçeklendirme anahtarını" `joblib` ile kaydediyoruz.

> **Neden Gerekli?** Eğer bu dosyaları kaydetmezsek, Notebook'u kapattığımızda modelin tüm öğrenmesi silinir ve yeni bir müşteri için tahmin yapamayız.

In [ ]:
import joblib

# Scaler nesnesini (ölçeklendirme anahtarını) kaydediyoruz
joblib.dump(scaler, 'scaler.pkl')

print("Ölçeklendirme anahtarı (scaler.pkl) başarıyla kaydedildi!")

## 🔮 8. Canlı Tahmin Denemesi (Prediction)
Bu bölümde, eğittiğimiz modeli ve kaydettiğimiz `scaler.pkl` dosyasını kullanarak tamamen yeni/hayali bir müşteri için tahminleme yapıyoruz. 
> **Süreç:** Veri yüklenir -> Scaler ile ölçeklenir -> Model olasılık hesaplar.

In [ ]:
import joblib
import tensorflow as tf
import numpy as np

# 1. Kaydettiğimiz Scaler ve Modeli geri yükleyelim
yuklenen_scaler = joblib.load('scaler.pkl')
yuklenen_model = tf.keras.models.load_model('../models/churn_modeli.keras')

# 2. Test setinden gerçek bir örnek alalım (Veya manuel veri girebiliriz)
# X_test zaten ölçeklenmiş olduğu için direkt modele verebiliriz.
ornek_index = 0
bir_musteri = X_test[ornek_index:ornek_index+1] 

# 3. Modelden tahmin isteyelim
tahmin_olasiligi = yuklenen_model.predict(bir_musteri)
sonuc_yuzdesi = tahmin_olasiligi[0][0] * 100

print("-" * 30)
print(f"Müşterinin Ayrılma (Churn) İhtimali: %{sonuc_yuzdesi:.2f}")

# 4. Karar verme (%50 eşik değeri)
if sonuc_yuzdesi > 50:
    print("⚠️ DİKKAT: Bu müşteri yüksek ihtimalle bizi bırakacak!")
else:
    print("✅ Müşteri sadık görünüyor, kalma ihtimali yüksek.")
print("-" * 30)

## 🔍 9. Tahmin Süreci Nasıl Çalışıyor? (Mantıksal Akış)

Yukarıdaki kod hücresini çalıştırdığımızda arka planda şu işlemler gerçekleşmektedir:

1.  **Hafızayı Geri Çağırma:** Kaydettiğimiz "yapay zeka beynini" (`.keras`) ve verileri doğru ölçekte küçülten "anahtarı" (`scaler.pkl`) sisteme yüklüyoruz.
2.  **Veri Hazırlığı:** Modelin daha önce hiç görmediği bir müşterinin verilerini alıyoruz.
3.  **Olasılık Hesaplama:** Model, müşterinin geçmiş alışkanlıklarına bakarak `0.0` (kalır) ile `1.0` (gider) arasında bir risk puanı üretir.
4.  **Karar:** Skor **%50'den büyükse** "Riskli", **küçükse** "Sadık" sonucu döndürülür.

## 📊 10. Model Performans Analizi (Hata Matrisi)

Modelin sadece %79 başarılı olduğunu bilmek yetmez; hangi konularda hata yaptığını da görmemiz gerekir. 
* **True Positive:** Gerçekten gidenleri doğru bildiğimiz sayı.
* **False Negative:** Gidecek olanlara "kalır" dediğimiz (en tehlikeli hata) sayı.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Tahminleri alalım
tahminler = model.predict(X_test)
y_pred = [1 if p > 0.5 else 0 for p in tahminler]

# 2. Matrisi oluşturalım
cm = confusion_matrix(y_test, y_pred)

# 3. Görselleştirelim
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Modelin Tahmini')
plt.ylabel('Gerçek Durum')
plt.title('Hata Matrisi (Confusion Matrix)')
plt.show()

## 🧐 11. Hata Matrisi Analizi: Grafik Bize Ne Anlatıyor?

Hata Matrisi (Confusion Matrix), modelimizin tahminlerinin ne kadar isabetli olduğunu dört farklı kategoride özetler:

1.  **Sol Üst (True Negative):** Modelin "Aboneliğini iptal etmeyecek" dediği ve gerçekten de iptal etmeyen sadık müşterilerdir.
2.  **Sağ Alt (True Positive):** Modelin "Aboneliğini iptal edecek" dediği ve gerçekten de iptal eden müşterilerdir.
3.  **Sağ Üst (False Positive):** Modelin "Gidecek" dediği ama aslında kalmaya devam eden müşterilerdir. (Hatalı alarm)
4.  **Sol Alt (False Negative):** Modelin "Kalacak" dediği ama aslında aboneliğini iptal eden müşterilerdir.
    * **Kritik Not:** Şirket için en riskli grup burasıdır; çünkü gidecek olan müşteriyi tahmin edemediğimiz için onlara özel kampanya yapma fırsatını kaçırmış oluruz.

> **Sonuç:** Modelimizin başarısını sadece genel doğruluk oranına (%79) bakarak değil, bu matris üzerindeki hata dağılımına bakarak değerlendirmeliyiz.